# Home Credit — PIS по группам, Python-прототип

**Цель:** на **полном наборе записей** получить по категориям дополнительного признака две статистики: число общих клиентов банка и партнёра (`N`) и сумму их бинарного `TARGET` (`defaults`). Рассчитать долю дефолтов, WoE и IV. Отдельный ноутбук, не требует C++, Bazel или gRPC.

**Модель конфиденциальности:** используем HMAC-псевдонимы для сопоставления, затем аддитивное гомоморфное шифрование Paillier для передачи сумм. **Это учебный прототип, а не криптографически безопасный двухсторонний Private Intersection-Sum.** Партнёр, сопоставляющий HMAC-псевдонимы, узнаёт, какие *его* клиенты совпали с банковскими. Банк получает только агрегаты и не получает от партнёра индивидуальные дополнительные признаки; HMAC-ключ используется обеими сторонами в демонстрации. Полноценный PIS требует протокола приватного сопоставления с защищённой агрегацией. Ниже есть честный аудит раскрываемой информации.

Данные `SK_ID_CURR` / `TARGET` реальные из Home Credit; синтетические номера телефона не нужны: здесь используем ID как условный общий ключ. Сами характеристики «телекома» первоначально собраны Home Credit — мы моделируем распределение прав доступа.

## 1. Установка библиотек

In [2]:
%pip -q install pandas pyarrow phe

Note: you may need to restart the kernel to use updated packages.


## 2. Входные файлы и выбор признака

Ожидаются `/content/bank_psi.parquet` и `/content/telecom_train.parquet` (как в ноутбуке коллеги). В банке должны быть `SK_ID_CURR, TARGET`, у партнёра — `SK_ID_CURR` и **выбранный признак**. Если в подготовленной телеком-таблице этого признака нет, укажи путь к исходной таблице с дополнительными признаками в `PARTNER_PATH` вместо `telecom_train.parquet`; склеивать с банковской таблицей здесь автоматически не будем.

In [3]:
from pathlib import Path
import time, secrets, hmac, hashlib
import numpy as np
import pandas as pd
from phe import paillier

BANK_PATH = Path('../data/bank_psi.parquet')
PARTNER_PATH = Path('../data/telecom_train.parquet')

# Можно изменить на 'DEF_30_CNT_SOCIAL_CIRCLE', 'OBS_30_CNT_SOCIAL_CIRCLE',
# 'DEF_60_CNT_SOCIAL_CIRCLE', 'OBS_60_CNT_SOCIAL_CIRCLE'.
FEATURE = 'DAYS_LAST_PHONE_CHANGE'

for p in (BANK_PATH, PARTNER_PATH):
    if not p.exists():
        raise FileNotFoundError(f'Нет файла: {p}. Загрузите его в Colab и поправьте путь.')

import pyarrow.parquet as pq
print('Банк:', pq.ParquetFile(BANK_PATH).metadata.num_rows, 'строк')
print('Партнёр:', pq.ParquetFile(PARTNER_PATH).metadata.num_rows, 'строк')
print('Поля партнёра:', pq.ParquetFile(PARTNER_PATH).schema_arrow.names)
if FEATURE not in pq.ParquetFile(PARTNER_PATH).schema_arrow.names:
    raise ValueError(
        f'В {PARTNER_PATH.name} отсутствует {FEATURE}. '
        'Нужна таблица партнёра с этим столбцом; поменяйте PARTNER_PATH.'
    )

Банк: 307491 строк
Партнёр: 307511 строк
Поля партнёра: ['SK_ID_CURR', 'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE', 'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE', 'DAYS_LAST_PHONE_CHANGE']


## 3. Загружаем **все** строки и строим бины

Пропуски — отдельная категория. Для `DAYS_LAST_PHONE_CHANGE` используем неотрицательную давность в днях; в Home Credit исходное число чаще отрицательное, а `0` / положительные аномалии нужно проверить. Для `DEF_*` — категории 0, 1, 2 и более.

In [4]:
bank = pd.read_parquet(BANK_PATH, columns=['SK_ID_CURR', 'TARGET']).dropna(subset=['SK_ID_CURR']).copy()
partner = pd.read_parquet(PARTNER_PATH, columns=['SK_ID_CURR', FEATURE]).dropna(subset=['SK_ID_CURR']).copy()

if bank.groupby('SK_ID_CURR')['TARGET'].nunique(dropna=True).gt(1).any():
    raise ValueError('У одного банковского ID несколько различных TARGET')
if partner.groupby('SK_ID_CURR')[FEATURE].nunique(dropna=True).gt(1).any():
    raise ValueError('У одного ID партнёра разные значения выбранного признака: нужна агрегация')
bank = bank.drop_duplicates('SK_ID_CURR').dropna(subset=['TARGET']).copy()
partner = partner.drop_duplicates('SK_ID_CURR').copy()
if not bank['TARGET'].isin([0,1]).all():
    raise ValueError('TARGET должен быть равен 0 или 1')
bank['TARGET'] = bank['TARGET'].astype('int64')

def assign_bins(series: pd.Series, feature: str) -> pd.Series:
    v = pd.to_numeric(series, errors='coerce')
    if feature == 'DAYS_LAST_PHONE_CHANGE':
        age = -v
        bins = pd.cut(age.where(age >= 0), [-0.001, 365, 1095, np.inf],
                      labels=['0–1 год', '1–3 года', 'более 3 лет'])
    else:
        bins = pd.cut(v.where(v >= 0), [-0.001, 0, 1, np.inf],
                      labels=['0', '1', '2+'])
    return bins.astype('object').fillna('Пропуск')

partner['bin'] = assign_bins(partner[FEATURE], FEATURE)
print('Клиентов банка:', len(bank))
print('Клиентов партнёра:', len(partner))
print('Бины партнёра:')
print(partner['bin'].value_counts(dropna=False).to_string())

Клиентов банка: 307491
Клиентов партнёра: 307511
Бины партнёра:
bin
более 3 лет    119393
1–3 года        95497
0–1 год         92620
Пропуск             1


## 4. Подготовка псевдонимов и зашифрованных банковских меток

HMAC предотвращает прямую передачу открытых ID, но **не скрывает факт совпадения от участника, который сравнивает токены**. Для скорости и практического запуска на полном объёме банковские `TARGET` сначала агрегируются по совпадающим HMAC-токенам в доверенной локальной демонстрационной среде, а затем шифруются только групповые итоги. Это **не** протокол, выполняющий защищённый JOIN. В настоящем двухстороннем PIS агрегация должна быть защищена *до* сопоставления; здесь это контрольная реализация вычисления нужных групповых статистик и работы Paillier.

In [5]:
t0 = time.perf_counter()
# В рабочей демонстрации обе стороны договорились об одном случайном HMAC-ключе.
# В настоящем PIS общий HMAC не заменяет PSI: сравнивающая сторона видит состав пересечения.
shared_key = secrets.token_bytes(32)

def token(value: object) -> str:
    canonical = str(int(value)).encode('utf-8')
    return hmac.new(shared_key, canonical, hashlib.sha256).hexdigest()

bank['token'] = [token(x) for x in bank['SK_ID_CURR']]
partner['token'] = [token(x) for x in partner['SK_ID_CURR']]

# В демонстрации выполняется локальное сопоставление: оно НЕ приватно.
joined = bank[['token','TARGET']].merge(
    partner[['token','bin']], on='token', how='inner', validate='one_to_one'
)
counts = joined.groupby('bin', sort=True).agg(
    N=('TARGET', 'size'), defaults=('TARGET', 'sum')
).reset_index()
counts['N'] = counts['N'].astype(int)
counts['defaults'] = counts['defaults'].astype(int)
print(f'Совпало {len(joined):,} клиентов | локальная подготовка: {time.perf_counter()-t0:.1f} с')
print('Банк не должен получать таблицу joined в реальной двухсторонней системе.')

Совпало 307,491 клиентов | локальная подготовка: 2.9 с
Банк не должен получать таблицу joined в реальной двухсторонней системе.


## 5. Paillier: шифрование групповых итогов

Демонстрируем **подлинное** аддитивно-гомоморфное шифрование, но **лишь поверх уже посчитанных локально агрегатов**. Это не обеспечивает приватность этапа сопоставления. Шифруем `N` и `defaults` каждой группы, выполняем на зашифрованных значениях сложение (контроль) и расшифровываем итог банковским секретным ключом.

In [6]:
t1 = time.perf_counter()
public_key, private_key = paillier.generate_paillier_keypair(n_length=2048)
crypted = []
for row in counts.itertuples(index=False):
    encrypted_n = public_key.encrypt(int(row.N))
    encrypted_defaults = public_key.encrypt(int(row.defaults))
    crypted.append((row.bin, encrypted_n, encrypted_defaults))

# Банк расшифровывает только суммы по группам
result_rows = [
    (name, private_key.decrypt(enc_n), private_key.decrypt(enc_def))
    for name, enc_n, enc_def in crypted
]
result = pd.DataFrame(result_rows, columns=['bin', 'N', 'defaults'])
result['default_rate'] = result['defaults'] / result['N'].replace(0, np.nan)
print(f'Шифрование / расшифрование Paillier: {time.perf_counter()-t1:.2f} с')
print(result.to_string(index=False))

# Проверка гомоморфного сложения зашифрованных значений
if crypted:
    total_ciphertext = sum((c[2] for c in crypted), public_key.encrypt(0))
    assert private_key.decrypt(total_ciphertext) == int(result['defaults'].sum())
    print('Проверка сложения под шифром прошла')

Шифрование / расшифрование Paillier: 1.43 с
        bin      N  defaults  default_rate
    0–1 год  92615      8870      0.095773
   1–3 года  95492      8488      0.088887
    Пропуск      1         0      0.000000
более 3 лет 119383      7466      0.062538
Проверка сложения под шифром прошла


## 6. Проверка на полной выборке и WoE / IV

**Важно:** WoE/IV считаются на раскрытых агрегатах. Сглаживание 0,5 в каждой группе предотвращает деление на ноль. По-настоящему защищённый протокол должен предоставлять банку именно такие `N` и `defaults` без выдачи ему отдельных ID и без раскрытия перечня совпавших ID телекому.

In [7]:
reference = (
    bank[['SK_ID_CURR', 'TARGET']]
    .merge(partner[['SK_ID_CURR','bin']], on='SK_ID_CURR', validate='one_to_one')
    .groupby('bin', sort=True).agg(N=('TARGET','size'), defaults=('TARGET','sum'))
    .reset_index()
)
check = result.merge(reference, on='bin', suffixes=('_encrypted', '_reference'), validate='one_to_one')
assert (check['N_encrypted'] == check['N_reference']).all()
assert (check['defaults_encrypted'] == check['defaults_reference']).all()
print('Эталонный JOIN: все групповые N и суммы TARGET совпали.')

result['nondefaults'] = result['N'] - result['defaults']
num_bins = len(result)
pos_total = result['defaults'].sum()
neg_total = result['nondefaults'].sum()
result['p_default'] = (result['defaults'] + 0.5) / (pos_total + 0.5 * num_bins)
result['p_nondefault'] = (result['nondefaults'] + 0.5) / (neg_total + 0.5 * num_bins)
result['WoE'] = np.log(result['p_nondefault'] / result['p_default'])
result['IV_contrib'] = (result['p_nondefault'] - result['p_default']) * result['WoE']
print(result[['bin', 'N', 'defaults', 'default_rate', 'WoE', 'IV_contrib']].to_string(index=False))
print('IV:', round(result['IV_contrib'].sum(), 5))

Эталонный JOIN: все групповые N и суммы TARGET совпали.
        bin      N  defaults  default_rate       WoE  IV_contrib
    0–1 год  92615      8870      0.095773 -0.187334    0.011435
   1–3 года  95492      8488      0.088887 -0.105138    0.003588
    Пропуск      1         0      0.000000 -1.333773    0.000020
более 3 лет 119383      7466      0.062538  0.274951    0.026169
IV: 0.04121


## 7. Что получено и что ещё НЕ реализовано

- **Получено:** полный объём Home Credit; категории дополнительного признака; точные `N`, суммы дефолтов, доли, WoE, IV; шифрование агрегатов на Paillier; проверка с открытым JOIN.
- **Не получено:** настоящий 2-party Private Intersection-Sum. В ячейке 4 есть локальное сопоставление, поэтому защищена лишь последующая передача агрегатов, а не вычисление пересечения. HMAC не исправляет этот недостаток.
- **Чтобы получить полноценный PIS**, потребуется криптографическая реализация приватного пересечения *с агрегацией* (например, проверенный PJC либо отдельный корректный MPC-протокол). Нельзя называть данный ноутбук готовым безопасным PIS в отчёте.
- **Риски даже у настоящего PIS:** малые группы и повторные запросы могут раскрывать информацию через полученные суммы, поэтому нужны пороги размера групп и правила публикации результатов.

**Для исходного кейса:** группы `DEF_30_CNT_SOCIAL_CIRCLE`, `DEF_60_CNT_SOCIAL_CIRCLE` и `DAYS_LAST_PHONE_CHANGE` позволяют получить содержательную таблицу по каждому признаку. Можно менять `FEATURE` и выполнять ноутбук заново.